In [1]:
import numpy as np
from sklearn.datasets import fetch_openml

In [4]:
# --- Hyperparameters ---
NUM_INPUT = 784
NUM_OUTPUT = 50
DT = 0.005         # 5 ms time step
T_MAX = 0.350      # 350 ms per image
EPOCHS = 1
LR = 0.05          # Learning rate

def train_snn():
    print("Loading MNIST training data via sklearn...")
    mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='liac-arff')
    
    # Use the first 60,000 images for training and normalize
    images = mnist.data[:60000] / 255.0
    
    # Initialize weights U(0, 1)
    weights = np.random.uniform(0, 1, (NUM_INPUT, NUM_OUTPUT))
    
    # Pre-allocate neuron states
    v_pre = np.zeros(NUM_INPUT)
    v_post = np.zeros(NUM_OUTPUT)
    n_adapt = np.zeros(NUM_OUTPUT)
    
    refractory_pre = np.zeros(NUM_INPUT)
    refractory_post = np.zeros(NUM_OUTPUT)
    wta_clamp = np.zeros(NUM_OUTPUT)

    print("Starting training...")
    for epoch in range(EPOCHS):
        for img_idx, img in enumerate(images):
            # Reset states for new image
            v_pre.fill(0.0)
            v_post.fill(0.0)
            
            # Input current proportional to pixel intensity
            I_in = img 
            
            for t in np.arange(0, T_MAX, DT):
                # --- Update Input Neurons (LIF) ---
                active_pre = refractory_pre <= 0
                v_pre[active_pre] += (DT / 0.03) * (-v_pre[active_pre] + I_in[active_pre] + 0.5)
                
                spikes_pre = v_pre >= 1.0
                v_pre[spikes_pre] = -1.0
                refractory_pre[spikes_pre] = 0.005
                refractory_pre -= DT
                
                # --- Update Output Neurons (ALIF) ---
                active_post = (refractory_post <= 0) & (wta_clamp <= 0)
                I_post = spikes_pre.astype(float) @ weights
                
                v_post[active_post] += (DT / 0.03) * (-v_post[active_post] + I_post[active_post] - n_adapt[active_post])
                n_adapt -= (DT / 1.0) * n_adapt
                
                spikes_post = v_post >= 1.0
                
                # --- Apply WTA and VDSP ---
                if spikes_post.any():
                    winner_idx = np.argmax(v_post) # Winner takes all
                    v_post[winner_idx] = 0.0
                    refractory_post[winner_idx] = 0.005
                    n_adapt[winner_idx] += 0.01
                    
                    # Clamp others
                    wta_clamp[:] = 0.010
                    wta_clamp[winner_idx] = 0.0
                    v_post[wta_clamp > 0] = 0.0
                    
                    # VDSP Weight Update for winner
                    v_pre_winner = v_pre
                    
                    # Potentiation (V_pre < 0)
                    pot_mask = v_pre_winner < 0
                    weights[pot_mask, winner_idx] += LR * (1.0 - weights[pot_mask, winner_idx]) * np.abs(v_pre_winner[pot_mask] + 1)
                    
                    # Depression (V_pre > 0)
                    dep_mask = v_pre_winner > 0
                    weights[dep_mask, winner_idx] -= LR * weights[dep_mask, winner_idx] * np.abs(v_pre_winner[dep_mask])
                    
                    # Clip weights (safety bound)
                    weights[:, winner_idx] = np.clip(weights[:, winner_idx], 0, 1)

                refractory_post -= DT
                wta_clamp -= DT
                
            if (img_idx + 1) % 1000 == 0:
                print(f"Epoch {epoch+1}, Processed {img_idx + 1} images")
                
    np.save('vdsp_weights.npy', weights)
    print("Training complete. Weights saved.")


In [5]:
train_snn()

Loading MNIST training data via sklearn...
Starting training...
Epoch 1, Processed 1000 images
Epoch 1, Processed 2000 images
Epoch 1, Processed 3000 images
Epoch 1, Processed 4000 images
Epoch 1, Processed 5000 images
Epoch 1, Processed 6000 images
Epoch 1, Processed 7000 images
Epoch 1, Processed 8000 images
Epoch 1, Processed 9000 images
Epoch 1, Processed 10000 images
Epoch 1, Processed 11000 images
Epoch 1, Processed 12000 images
Epoch 1, Processed 13000 images
Epoch 1, Processed 14000 images
Epoch 1, Processed 15000 images
Epoch 1, Processed 16000 images
Epoch 1, Processed 17000 images
Epoch 1, Processed 18000 images
Epoch 1, Processed 19000 images
Epoch 1, Processed 20000 images
Epoch 1, Processed 21000 images
Epoch 1, Processed 22000 images
Epoch 1, Processed 23000 images
Epoch 1, Processed 24000 images
Epoch 1, Processed 25000 images
Epoch 1, Processed 26000 images
Epoch 1, Processed 27000 images
Epoch 1, Processed 28000 images
Epoch 1, Processed 29000 images
Epoch 1, Processe